# Convert our Images into Text

In [1]:
import torch
from transformers import AutoProcessor, Blip2ForConditionalGeneration
import torch
import os
import requests
from PIL import Image
from io import BytesIO
from pathlib import Path
import pandas as pd
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed

In [2]:
print(f"CUDA available?: {torch.cuda.is_available()}")
print(f"Device Name: {torch.cuda.get_device_name()}")
device = "cuda:0"
torch.cuda.set_device(device)

CUDA available?: True
Device Name: AMD Radeon RX 9060 XT


## Demo

In [3]:
processor = AutoProcessor.from_pretrained("Salesforce/blip2-opt-2.7b")
model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b", 
    dtype=torch.float16
).to(device) 

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [4]:
img_url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
image = Image.open(requests.get(img_url, stream=True).raw).convert("RGB")

In [5]:
def generate_caption(image):
    inputs = processor(image, return_tensors="pt").to(device, torch.float16)

    generated_ids = model.generate(**inputs, max_new_tokens=30)
    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return generated_text

In [6]:
generate_caption(image)

/home/maharshii/miniconda3/envs/mlgeneral/lib/python3.10/site-packages/transformers/integrations/sdpa_attention.py:96: UserWarning: Flash Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:316.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
/home/maharshii/miniconda3/envs/mlgeneral/lib/python3.10/site-packages/transformers/integrations/sdpa_attention.py:96: UserWarning: Mem Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:373.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


'a cat walking on snow in an enclosure'

## Convert Images into Text

### Load Datasets

In [43]:
data_dir = Path("../cleaned")  # change if your data directory is elsewhere

datasets = []

for file_name in data_dir.rglob("*.csv"):
    if file_name.stem.endswith("_images"):
        df = pd.read_csv(str(file_name))
        df._name = file_name.name[:-11]  # or file_name.stem, or str(file_name)
        datasets.append(df)

### Cleaning Datasets

In [44]:
for df in datasets:
    df["subreddit"] = df._name  # Assign subreddit name from file name to each dataframe
    df.drop(columns=["media_id", "image_type", "image_source"], inplace=True) # Drop media_id

In [9]:
def parse_image(image_url, headers={"User-Agent": "Mozilla/5.0"}):
    resp = requests.get(image_url, headers=headers, timeout=5)
    resp.raise_for_status()
    image = Image.open(BytesIO(resp.content)).convert("RGBA")
    return image

In [10]:
def url_is_ok(url):
    try:
        parse_image(url)
        return True
    except Exception:
        return False

In [13]:
cpu_count = os.cpu_count() or 4
print("CPU cores:", cpu_count)

# Good starting points for IO-bound (network) work:
max_workers = cpu_count * 5   # or * 10 if your network / API can handle it
print("Suggested max_workers:", max_workers)

CPU cores: 12
Suggested max_workers: 60


In [14]:
cleaned_datasets = []
total_datasets = len(datasets)

for i, df in enumerate(datasets, start=1):
    original_len = len(df)
    bad_idx = []

    # run URL checks in parallel
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_idx = {
            executor.submit(url_is_ok, url): idx
            for idx, url in df["image_url"].items()
        }
        for future in as_completed(future_to_idx):
            idx = future_to_idx[future]
            ok = future.result()
            if not ok:
                bad_idx.append(idx)

    df = df.drop(index=bad_idx)
    cleaned_datasets.append(df)

    removed = len(bad_idx)
    pct_removed = (removed / original_len * 100) if original_len > 0 else 0.0

    print(f"Dataset {i}/{total_datasets}: removed {removed} images "
          f"({pct_removed:.2f}% removed).")
    print(f"Datasets left to filter through: {total_datasets - i}")

Dataset 1/78: removed 1 images (12.50% removed).
Datasets left to filter through: 77
Dataset 2/78: removed 6 images (54.55% removed).
Datasets left to filter through: 76
Dataset 3/78: removed 0 images (0.00% removed).
Datasets left to filter through: 75
Dataset 4/78: removed 14 images (2.97% removed).
Datasets left to filter through: 74
Dataset 5/78: removed 5 images (38.46% removed).
Datasets left to filter through: 73
Dataset 6/78: removed 48 images (6.71% removed).
Datasets left to filter through: 72
Dataset 7/78: removed 0 images (0.00% removed).
Datasets left to filter through: 71
Dataset 8/78: removed 0 images (0.00% removed).
Datasets left to filter through: 70
Dataset 9/78: removed 20 images (9.17% removed).
Datasets left to filter through: 69
Dataset 10/78: removed 3 images (15.79% removed).
Datasets left to filter through: 68
Dataset 11/78: removed 6 images (3.49% removed).
Datasets left to filter through: 67
Dataset 12/78: removed 4 images (40.00% removed).
Datasets left to 

/home/maharshii/miniconda3/envs/mlgeneral/lib/python3.10/site-packages/PIL/Image.py:3432: DecompressionBombWarning: Image size (114738774 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Dataset 24/78: removed 29 images (5.82% removed).
Datasets left to filter through: 54
Dataset 25/78: removed 63 images (22.42% removed).
Datasets left to filter through: 53
Dataset 26/78: removed 1 images (100.00% removed).
Datasets left to filter through: 52
Dataset 27/78: removed 10 images (3.40% removed).
Datasets left to filter through: 51
Dataset 28/78: removed 1 images (14.29% removed).
Datasets left to filter through: 50
Dataset 29/78: removed 12 images (23.08% removed).
Datasets left to filter through: 49
Dataset 30/78: removed 43 images (9.01% removed).
Datasets left to filter through: 48


/home/maharshii/miniconda3/envs/mlgeneral/lib/python3.10/site-packages/PIL/Image.py:3432: DecompressionBombWarning: Image size (154716504 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Dataset 31/78: removed 10 images (1.81% removed).
Datasets left to filter through: 47
Dataset 32/78: removed 152 images (26.81% removed).
Datasets left to filter through: 46
Dataset 33/78: removed 5 images (35.71% removed).
Datasets left to filter through: 45


...

Dataset 58/78: removed 18 images (4.00% removed).
Datasets left to filter through: 20
Dataset 59/78: removed 4 images (57.14% removed).
Datasets left to filter through: 19
Dataset 60/78: removed 42 images (13.38% removed).
Datasets left to filter through: 18
Dataset 61/78: removed 1 images (2.33% removed).
Datasets left to filter through: 17
Dataset 62/78: removed 2 images (1.54% removed).
Datasets left to filter through: 16
Dataset 63/78: removed 5 images (4.55% removed).
Datasets left to filter through: 15
Dataset 64/78: removed 4 images (4.21% removed).
Datasets left to filter through: 14
Dataset 65/78: removed 0 images (0.00% removed).
Datasets left to filter through: 13
Dataset 66/78: removed 13 images (2.50% removed).
Datasets left to filter through: 12
Dataset 67/78: removed 98 images (15.93% removed).
Datasets left to filter through: 11
Dataset 68/78: removed 21 images (1.64% removed).
Datasets left to filter through: 10


/home/maharshii/miniconda3/envs/mlgeneral/lib/python3.10/site-packages/PIL/Image.py:3432: DecompressionBombWarning: Image size (108000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Dataset 69/78: removed 2 images (1.44% removed).
Datasets left to filter through: 9
Dataset 70/78: removed 41 images (2.45% removed).
Datasets left to filter through: 8


/home/maharshii/miniconda3/envs/mlgeneral/lib/python3.10/site-packages/PIL/Image.py:3432: DecompressionBombWarning: Image size (96000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/home/maharshii/miniconda3/envs/mlgeneral/lib/python3.10/site-packages/PIL/Image.py:3432: DecompressionBombWarning: Image size (161678160 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


Dataset 71/78: removed 34 images (6.55% removed).
Datasets left to filter through: 7
Dataset 72/78: removed 2 images (0.78% removed).
Datasets left to filter through: 6
Dataset 73/78: removed 1 images (14.29% removed).
Datasets left to filter through: 5
Dataset 74/78: removed 0 images (0.00% removed).
Datasets left to filter through: 4
Dataset 75/78: removed 3 images (3.33% removed).
Datasets left to filter through: 3
Dataset 76/78: removed 5 images (22.73% removed).
Datasets left to filter through: 2
Dataset 77/78: removed 77 images (13.73% removed).
Datasets left to filter through: 1
Dataset 78/78: removed 9 images (1.88% removed).
Datasets left to filter through: 0


### Images -> Text Coversion

In [18]:
cpu_count = os.cpu_count() or 12
max_workers_fetch = cpu_count * 4      # ~48 workers on your CPU
max_workers_caption = 1                # GPU-bound, keep serial

captioned_datasets = []
total_datasets = len(cleaned_datasets)

for i, df in enumerate(cleaned_datasets, start=1):
    captions = {}

    # parallelize: fetch image + caption, small worker count (GPU-bound)
    def fetch_and_caption(idx, url):
        img = parse_image(url)          # network + PIL
        cap = generate_caption(img)     # GPU caption
        return idx, cap

    with ThreadPoolExecutor(max_workers=max_workers_caption) as executor:
        futures = [
            executor.submit(fetch_and_caption, idx, url)
            for idx, url in df["image_url"].items()
        ]

        for future in as_completed(futures):
            try:
                idx, cap = future.result()
            except Exception:
                # if captioning fails, store None (or "" if you prefer)
                idx, cap = None, None
            if idx is not None:
                captions[idx] = cap

    df["caption"] = df.index.map(captions.get)
    captioned_datasets.append(df)

    print(f"[CAPTION] Dataset {i}/{total_datasets} captioned "
          f"({total_datasets - i} datasets left).")

[CAPTION] Dataset 1/78 captioned (77 datasets left).
[CAPTION] Dataset 2/78 captioned (76 datasets left).
[CAPTION] Dataset 3/78 captioned (75 datasets left).
[CAPTION] Dataset 4/78 captioned (74 datasets left).
[CAPTION] Dataset 5/78 captioned (73 datasets left).
[CAPTION] Dataset 6/78 captioned (72 datasets left).
[CAPTION] Dataset 7/78 captioned (71 datasets left).
[CAPTION] Dataset 8/78 captioned (70 datasets left).
[CAPTION] Dataset 9/78 captioned (69 datasets left).
[CAPTION] Dataset 10/78 captioned (68 datasets left).
[CAPTION] Dataset 11/78 captioned (67 datasets left).
[CAPTION] Dataset 12/78 captioned (66 datasets left).
[CAPTION] Dataset 13/78 captioned (65 datasets left).
[CAPTION] Dataset 14/78 captioned (64 datasets left).
[CAPTION] Dataset 15/78 captioned (63 datasets left).
[CAPTION] Dataset 16/78 captioned (62 datasets left).
[CAPTION] Dataset 17/78 captioned (61 datasets left).
[CAPTION] Dataset 18/78 captioned (60 datasets left).
[CAPTION] Dataset 19/78 captioned (59

/home/maharshii/miniconda3/envs/mlgeneral/lib/python3.10/site-packages/PIL/Image.py:3432: DecompressionBombWarning: Image size (114738774 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


[CAPTION] Dataset 24/78 captioned (54 datasets left).
[CAPTION] Dataset 25/78 captioned (53 datasets left).
[CAPTION] Dataset 26/78 captioned (52 datasets left).
[CAPTION] Dataset 27/78 captioned (51 datasets left).
[CAPTION] Dataset 28/78 captioned (50 datasets left).
[CAPTION] Dataset 29/78 captioned (49 datasets left).
[CAPTION] Dataset 30/78 captioned (48 datasets left).


/home/maharshii/miniconda3/envs/mlgeneral/lib/python3.10/site-packages/PIL/Image.py:3432: DecompressionBombWarning: Image size (154716504 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


[CAPTION] Dataset 31/78 captioned (47 datasets left).
[CAPTION] Dataset 32/78 captioned (46 datasets left).
[CAPTION] Dataset 33/78 captioned (45 datasets left).


...

[CAPTION] Dataset 58/78 captioned (20 datasets left).
[CAPTION] Dataset 59/78 captioned (19 datasets left).
[CAPTION] Dataset 60/78 captioned (18 datasets left).
[CAPTION] Dataset 61/78 captioned (17 datasets left).
[CAPTION] Dataset 62/78 captioned (16 datasets left).
[CAPTION] Dataset 63/78 captioned (15 datasets left).
[CAPTION] Dataset 64/78 captioned (14 datasets left).
[CAPTION] Dataset 65/78 captioned (13 datasets left).
[CAPTION] Dataset 66/78 captioned (12 datasets left).
[CAPTION] Dataset 67/78 captioned (11 datasets left).
[CAPTION] Dataset 68/78 captioned (10 datasets left).


/home/maharshii/miniconda3/envs/mlgeneral/lib/python3.10/site-packages/PIL/Image.py:3432: DecompressionBombWarning: Image size (108000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


[CAPTION] Dataset 69/78 captioned (9 datasets left).
[CAPTION] Dataset 70/78 captioned (8 datasets left).


/home/maharshii/miniconda3/envs/mlgeneral/lib/python3.10/site-packages/PIL/Image.py:3432: DecompressionBombWarning: Image size (161678160 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/home/maharshii/miniconda3/envs/mlgeneral/lib/python3.10/site-packages/PIL/Image.py:3432: DecompressionBombWarning: Image size (96000000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


[CAPTION] Dataset 71/78 captioned (7 datasets left).
[CAPTION] Dataset 72/78 captioned (6 datasets left).
[CAPTION] Dataset 73/78 captioned (5 datasets left).
[CAPTION] Dataset 74/78 captioned (4 datasets left).
[CAPTION] Dataset 75/78 captioned (3 datasets left).
[CAPTION] Dataset 76/78 captioned (2 datasets left).
[CAPTION] Dataset 77/78 captioned (1 datasets left).
[CAPTION] Dataset 78/78 captioned (0 datasets left).


### Save Data Locally

In [34]:
data_dir = Path("../data")

csv_files = [
    str(file_name)[:8] + "cleaned/" + str(file_name)[8:] for file_name in data_dir.rglob("*.csv")
    if file_name.stem.endswith("_images")
]

In [ ]:
for df in datasets:
    df.to_csv(f"../cleaned/{df._name}_images.csv", index=False)
    print(f"Saved: ../cleaned/{df._name}_images.csv")

Saved: ../data/cleaned/sports_images.csv
Saved: ../data/cleaned/maybemaybemaybe_images.csv
Saved: ../data/cleaned/EatCheapAndHealthy_images.csv
Saved: ../data/cleaned/HistoryMemes_images.csv
Saved: ../data/cleaned/videos_images.csv
Saved: ../data/cleaned/pics_images.csv
Saved: ../data/cleaned/books_images.csv
Saved: ../data/cleaned/Jokes_images.csv
Saved: ../data/cleaned/Bitcoin_images.csv
Saved: ../data/cleaned/Showerthoughts_images.csv
Saved: ../data/cleaned/oddlysatisfying_images.csv
Saved: ../data/cleaned/history_images.csv
Saved: ../data/cleaned/cars_images.csv
Saved: ../data/cleaned/PremierLeague_images.csv
Saved: ../data/cleaned/StockMarket_images.csv
Saved: ../data/cleaned/Daytrading_images.csv
Saved: ../data/cleaned/MadeMeSmile_images.csv
Saved: ../data/cleaned/mildlyinfuriating_images.csv
Saved: ../data/cleaned/ThriftStoreHauls_images.csv
Saved: ../data/cleaned/dadjokes_images.csv
Saved: ../data/cleaned/foodhacks_images.csv
Saved: ../data/cleaned/unitedkingdom_images.csv
Save

In [37]:
captioned_datasets[0]

,subreddit,post_id,comment_id,image_index,image_url,image_source,image_type,media_id,caption
0,NaN,1nn83c1,nfs6rbs,0,https://i.imgur.com/fEiAnK9.png,comment_text,embedded_link,NaN,a table with the names of the players on the y...
1,NaN,1nhmlbz,ned802x,0,https://i.redd.it/zzvcc5ox6qsy.jpg,comment_text,embedded_link,NaN,the office is a great show
2,NaN,1n4dwk2,nblb7ty,0,https://i.imgur.com/CKp1Rup.png)...,comment_text,embedded_link,NaN,a computer screen showing a video of a soccer ...
3,NaN,1n4dwk2,nblv56f,0,https://i.imgur.com/KriiYZS.png).,comment_text,embedded_link,NaN,a video game screen showing a football field
5,NaN,1mymugi,naj7o7o,0,https://i.imgur.com/UpPrvIp.jpeg),comment_text,embedded_link,NaN,a black and white photo of a man in a suit
6,NaN,1mn7qem,n865828,0,https://i.imgur.com/S0eSAh6.gif),comment_text,embedded_link,NaN,a man with a caption that says what the fuck d...
7,NaN,1mjdw6b,n7b37iz,0,https://i.imgur.com/HJ1lIFs.png,comment_text,embedded_link,NaN,a man laying in bed with his eyes closed
